In [1]:
!nvidia-smi

Thu Oct  8 07:07:14 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   52C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
from numba import cuda

device = cuda.get_current_device()

print("Compute Capability:", device.compute_capability)
print("Max Threads Per Block:", device.MAX_THREADS_PER_BLOCK)

Compute Capability: ComputeCapability(major=7, minor=5)
Max Threads Per Block: 1024


In [4]:
import numpy as np
import time
from numba import cuda

# Вычисление на CPU
def double_cpu_pure(arr):
    out = np.empty_like(arr)
    for i in range(len(arr)):
        out[i] = arr[i] * 2.0
    return out

# Вычисление на GPU
@cuda.jit
def double_gpu_kernel(d_in, d_out):
    idx = cuda.grid(1)
    if idx < d_in.shape[0]:
        d_out[idx] = d_in[idx] * 2.0

In [5]:
def benchmark_run(N, student_id_seed):
    np.random.seed(student_id_seed)
    h_in = np.random.rand(N).astype(np.float32)

    # CPU
    t0 = time.perf_counter()
    double_cpu_pure(h_in)
    cpu_time_ms = (time.perf_counter() - t0) * 1000.0

    threads_per_block = 256
    blocks_per_grid = (N + threads_per_block - 1) // threads_per_block

    # Прогрев: компиляция GPU-функции до измерений
    d_warmup = cuda.to_device(np.ones(10, dtype=np.float32))
    double_gpu_kernel[1, 32](d_warmup, d_warmup)
    cuda.synchronize()

    # GPU: передача данных + вычисление + возврат результата
    t0 = time.perf_counter()
    d_in = cuda.to_device(h_in)
    d_out = cuda.device_array_like(d_in)
    double_gpu_kernel[blocks_per_grid, threads_per_block](d_in, d_out)
    h_out = d_out.copy_to_host()
    cuda.synchronize()
    gpu_total_ms = (time.perf_counter() - t0) * 1000.0

    # GPU: запуск вычисления, данные уже в видеопамяти
    t0 = time.perf_counter()
    double_gpu_kernel[blocks_per_grid, threads_per_block](d_in, d_out)
    cuda.synchronize()
    gpu_compute_ms = (time.perf_counter() - t0) * 1000.0

    return cpu_time_ms, gpu_total_ms, gpu_compute_ms

In [6]:
student_seed = int(input("Введите свой числовой Student ID: "))

sizes = [
    100, 1000, 10000, 100000,
    1000000, 5000000, 10000000
]

results = []
CROSSOVER_N = None

print("\nN | CPU (ms) | GPU Total (ms) | GPU Kernel (ms) | Winner")
print("-" * 75)

for N in sizes:
    cpu, total, kernel = benchmark_run(N, student_seed)
    winner = "GPU" if total < cpu else "CPU"

    results.append((N, cpu, total, kernel, winner))

    if CROSSOVER_N is None and total < cpu:
        CROSSOVER_N = N

    print(
        f"{N:,} | {cpu:.3f} | {total:.3f} | "
        f"{kernel:.3f} | {winner}"
    )

print("\nCrossover Point:", CROSSOVER_N)

Введите свой числовой Student ID: 230103222

N | CPU (ms) | GPU Total (ms) | GPU Kernel (ms) | Winner
---------------------------------------------------------------------------


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 4 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 40 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


100 | 0.064 | 1.592 | 0.153 | CPU
1,000 | 0.525 | 0.737 | 0.082 | CPU
10,000 | 5.502 | 0.682 | 0.089 | GPU
100,000 | 53.298 | 0.902 | 0.092 | GPU
1,000,000 | 521.912 | 3.426 | 0.151 | GPU
5,000,000 | 1468.080 | 17.530 | 0.327 | GPU
10,000,000 | 2565.236 | 24.518 | 0.454 | GPU

Crossover Point: 10000


In [7]:
import numpy as np
from numba import cuda

@cuda.jit
def defective_kernel(d_arr):
    idx = cuda.grid(1)
    d_arr[idx] = 999.0  # Проверка границы специально отсутствует

N = 1000
h_arr = np.zeros(N, dtype=np.float32)
d_arr = cuda.to_device(h_arr)

threads_per_block = 256
blocks_per_grid = 4

print("Launching defective kernel with 1024 threads on 1000-element array...")

defective_kernel[blocks_per_grid, threads_per_block](d_arr)
cuda.synchronize()

print("Execution finished (or did it crash?). Checking host copy...")

h_result = d_arr.copy_to_host()
print(f"Elements processed: {len(h_result)}. Last element: {h_result[-1]}")

Launching defective kernel with 1024 threads on 1000-element array...
Execution finished (or did it crash?). Checking host copy...
Elements processed: 1000. Last element: 999.0


In [8]:
import numpy as np
from numba import cuda

@cuda.jit
def inspect_threads_kernel(log_block, log_thread, log_global, log_active, N):
    t_idx = cuda.threadIdx.x
    b_idx = cuda.blockIdx.x
    g_idx = cuda.grid(1)

    if g_idx < log_block.shape[0]:
        log_block[g_idx] = b_idx
        log_thread[g_idx] = t_idx
        log_global[g_idx] = g_idx
        log_active[g_idx] = 1 if g_idx < N else 0

N = 37
threads_per_block = 8
blocks_per_grid = (N + threads_per_block - 1) // threads_per_block
total_threads = blocks_per_grid * threads_per_block

log_b = cuda.device_array(total_threads, dtype=np.int32)
log_t = cuda.device_array(total_threads, dtype=np.int32)
log_g = cuda.device_array(total_threads, dtype=np.int32)
log_a = cuda.device_array(total_threads, dtype=np.int32)

inspect_threads_kernel[blocks_per_grid, threads_per_block](
    log_b, log_t, log_g, log_a, N
)
cuda.synchronize()

# Копируем записи на CPU для вывода
host_b = log_b.copy_to_host()
host_t = log_t.copy_to_host()
host_g = log_g.copy_to_host()
host_a = log_a.copy_to_host()

print(f"{'Global ID':<10} | {'Block ID':<10} | {'Thread ID':<10} | Status")
print("-" * 60)

for i in range(32, 40):
    status = "ACTIVE" if host_a[i] == 1 else "IDLE (GUARDED)"
    print(f"{host_g[i]:<10} | {host_b[i]:<10} | {host_t[i]:<10} | {status}")

Global ID  | Block ID   | Thread ID  | Status
------------------------------------------------------------
32         | 4          | 0          | ACTIVE
33         | 4          | 1          | ACTIVE
34         | 4          | 2          | ACTIVE
35         | 4          | 3          | ACTIVE
36         | 4          | 4          | ACTIVE
37         | 4          | 5          | IDLE (GUARDED)
38         | 4          | 6          | IDLE (GUARDED)
39         | 4          | 7          | IDLE (GUARDED)


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 5 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


In [9]:
import numpy as np
import time
from numba import cuda

@cuda.jit
def naive_sum_kernel(d_arr, d_total):
    idx = cuda.grid(1)
    if idx < d_arr.shape[0]:
        d_total[0] += d_arr[idx]

@cuda.jit
def atomic_sum_kernel(d_arr, d_total):
    idx = cuda.grid(1)
    if idx < d_arr.shape[0]:
        cuda.atomic.add(d_total, 0, d_arr[idx])

N = 50_000
threads_per_block = 256
blocks_per_grid = (N + threads_per_block - 1) // threads_per_block

d_arr = cuda.to_device(np.ones(N, dtype=np.float32))
zero = np.zeros(1, dtype=np.float32)
d_total = cuda.to_device(zero)

# Прогрев обоих вариантов: компиляция исключена из измерений
for kernel in (naive_sum_kernel, atomic_sum_kernel):
    d_total.copy_to_device(zero)
    kernel[blocks_per_grid, threads_per_block](d_arr, d_total)
    cuda.synchronize()

def measure_kernel(kernel):
    # Перед каждым запуском обнуляем сумму
    d_total.copy_to_device(zero)
    cuda.synchronize()

    start = time.perf_counter()
    kernel[blocks_per_grid, threads_per_block](d_arr, d_total)
    cuda.synchronize()
    elapsed_ms = (time.perf_counter() - start) * 1000.0

    result = float(d_total.copy_to_host()[0])
    return result, elapsed_ms

print("Trial | Target Sum | Naive Output | Lost Updates")
print("-" * 60)

naive_times = []

for trial in range(1, 6):
    result, elapsed = measure_kernel(naive_sum_kernel)
    naive_times.append(elapsed)
    lost_updates = N - result

    print(
        f"{trial} | {N:.1f} | "
        f"{result:.1f} | {lost_updates:.1f}"
    )

# Пять измерений atomic для сравнения средних времён
atomic_times = []
atomic_outputs = []

for trial in range(5):
    result, elapsed = measure_kernel(atomic_sum_kernel)
    atomic_outputs.append(result)
    atomic_times.append(elapsed)

print("\nAtomic Sum Output:", atomic_outputs[-1])
print(
    "All atomic results equal 50000.0:",
    "Yes" if all(x == N for x in atomic_outputs) else "No"
)
print(f"Naive Kernel Time (mean of 5): {np.mean(naive_times):.3f} ms")
print(f"Atomic Kernel Time (mean of 5): {np.mean(atomic_times):.3f} ms")

Trial | Target Sum | Naive Output | Lost Updates
------------------------------------------------------------
1 | 50000.0 | 2.0 | 49998.0
2 | 50000.0 | 2.0 | 49998.0
3 | 50000.0 | 2.0 | 49998.0
4 | 50000.0 | 2.0 | 49998.0
5 | 50000.0 | 2.0 | 49998.0

Atomic Sum Output: 50000.0
All atomic results equal 50000.0: Yes
Naive Kernel Time (mean of 5): 0.153 ms
Atomic Kernel Time (mean of 5): 0.297 ms


In [10]:
import math
import numpy as np
from numba import cuda

@cuda.jit
def radial_vignette_2d(d_canvas, width, height, center_x, center_y):
    x, y = cuda.grid(2)

    if x < width and y < height:
        dx = float(x - center_x)
        dy = float(y - center_y)

        dist = math.sqrt(dx * dx + dy * dy)
        max_dist = math.sqrt(float(center_x**2 + center_y**2))

        intensity = 1.0 - dist / max_dist
        if intensity < 0.0:
            intensity = 0.0

        d_canvas[y, x] = intensity

WIDTH, HEIGHT = 1024, 1024

canvas = np.zeros((HEIGHT, WIDTH), dtype=np.float32)
d_canvas = cuda.to_device(canvas)

threads_2d = (16, 16)
blocks_2d = (
    (WIDTH + threads_2d[0] - 1) // threads_2d[0],
    (HEIGHT + threads_2d[1] - 1) // threads_2d[1]
)

radial_vignette_2d[blocks_2d, threads_2d](
    d_canvas, WIDTH, HEIGHT, WIDTH // 2, HEIGHT // 2
)
cuda.synchronize()

result_canvas = d_canvas.copy_to_host()

print("Matrix shape:", result_canvas.shape)
print("Blocks in X:", blocks_2d[0])
print("Blocks in Y:", blocks_2d[1])
print("Total Blocks:", blocks_2d[0] * blocks_2d[1])
print(f"Center pixel (512, 512): {result_canvas[512, 512]:.3f}")
print(f"Corner pixel (0, 0): {result_canvas[0, 0]:.3f}")

Matrix shape: (1024, 1024)
Blocks in X: 64
Blocks in Y: 64
Total Blocks: 4096
Center pixel (512, 512): 1.000
Corner pixel (0, 0): 0.000


In [12]:
import hashlib

def generate_lab_checksum(student_id_str, d_canvas, exp_table_crossover):
    h = hashlib.sha256()
    h.update(student_id_str.strip().encode("utf-8"))

    gpu_name = cuda.get_current_device().name
    if isinstance(gpu_name, str):
        gpu_name = gpu_name.encode("utf-8")
    h.update(gpu_name)

    h.update(d_canvas.copy_to_host()[:10, :10].tobytes())
    h.update(str(exp_table_crossover).encode("utf-8"))

    digest = h.hexdigest()[:16].upper()
    print("=" * 60)
    print(f"OFFICIAL VERIFICATION CHECKSUM TOKEN: {digest}")
    print("=" * 60)
    return digest

STUDENT_ID = "230103222"
CROSSOVER_N = 10000

generate_lab_checksum(STUDENT_ID, d_canvas, CROSSOVER_N)

OFFICIAL VERIFICATION CHECKSUM TOKEN: 001EDA7B6529C8AE


'001EDA7B6529C8AE'